# 3LC Low-Light: hyperparameter sweep on Kaggle GPU (exploration only)

This notebook finds the training settings to use in the **3LC plugin's Train tab**. It trains exactly like the plugin does and scores the val set exactly like Kaggle scores the test set. It does not make a submission.

**Rules that apply to this notebook (host answers in the competition Discussion):**
- Exploratory training outside the plugin is allowed. **Every submission must come from a plugin Run.** Do not submit anything this notebook makes.
- Keep this notebook **private**. Do not save the starter kit as a Kaggle dataset. The notebook keeps the images in `/tmp`, so they do not go into the saved output.
- Do not change labels here. Label edits belong in the 3LC Dashboard.

**What it copies from the plugin (v1.2.15):** ultralytics 8.4.66, the official `yolo11n.pt` (sha256 pinned), `imgsz=640`, `pretrained=True`, batch 16, seed 0, patience 100, and only arguments the plugin accepts (bounds checked).

**How it scores:** pycocotools bbox AP at IoU 0.5 with at most 100 boxes per image, averaged over classes, the same as the Kaggle metric. Predictions use the plugin's Predict settings: imgsz 640, `max_det=300`, values clamped to [0, 1]. It scores both `best.pt` and `last.pt` at several Predict `conf` values, on the full val set and on val without the 14 images that duplicate train scenes.

## How to run
1. Kaggle: **Create → New Notebook → File → Import Notebook**, upload this file.
2. Settings panel: **Accelerator = GPU T4 x2**, **Internet = On** (needs a phone-verified account).
3. Set `STAGE` in the first code cell (1 for the first run).
4. **Save Version → Save & Run All (Commit)**. It runs in the background for up to 12 hours.
5. When the version finishes, open it and copy everything below `=== PASTE EVERYTHING BELOW ===` from the last cell's output.

Stage 1 uses about 7–10 GPU hours of your 30-hour weekly quota.

In [ ]:
# ---- Settings -------------------------------------------------------------
import time
NB_START = time.time()

STAGE = 1             # 1 = screening (40 epochs per config). 2 = long runs, after Stage 1 results.
TIME_BUDGET_H = 11.0  # Kaggle stops sessions at 12 h; runs stop cleanly before this.
CONFS = [0.001, 0.01, 0.05, 0.1, 0.25]   # Predict-tab "conf" values to score (plugin default is 0.25)

# Plugin defaults (trainer.py _EXPOSED): batch 16, patience 100; ultralytics seed 0.
COMMON = dict(batch=16, patience=100, seed=0, deterministic=True)

SGD_COS = dict(optimizer="SGD", lr0=0.01, cos_lr=True)
STAGES = {
    1: [  # 40 epochs each, most important first. 'plugin' = what to type in the Train tab.
        ("s1_default",       "Epochs 40; all other fields default (Optimizer Auto = AdamW, lr0 ignored)", dict(epochs=40)),
        ("s1_sgd_cos",       "Epochs 40, Optimizer SGD, lr0 0.01; Extra args: cos_lr=True",               dict(epochs=40, **SGD_COS)),
        ("s1_adamw_cos",     "Epochs 40, Optimizer AdamW, lr0 0.001; Extra args: cos_lr=True",            dict(epochs=40, optimizer="AdamW", lr0=0.001, cos_lr=True)),
        ("s1_sgd_cos_seed1", "Noise check only: same as s1_sgd_cos with seed=1",                          dict(epochs=40, **SGD_COS, seed=1)),
        ("s1_sgd_cos_bright","s1_sgd_cos + Extra args: hsv_v=0.6 mixup=0.1",                              dict(epochs=40, **SGD_COS, hsv_v=0.6, mixup=0.1)),
        ("s1_sgd_cos_scale", "s1_sgd_cos + Extra args: scale=0.7",                                        dict(epochs=40, **SGD_COS, scale=0.7)),
        ("s1_sgd_cos_ms",    "s1_sgd_cos + Extra args: multi_scale=0.25 (a fraction, NOT true)",          dict(epochs=40, **SGD_COS, multi_scale=0.25)),
        ("s1_sgd_cos_cls",   "s1_sgd_cos + Extra args: cls=1.0",                                          dict(epochs=40, **SGD_COS, cls=1.0)),
    ],
    2: [  # long runs; STAGE2_OVERRIDES is added to both
        ("s2_auto_150",    "Epochs 150, Optimizer Auto (= MuSGD at >= 108 epochs)", dict(epochs=150)),
        ("s2_sgd_cos_150", "Epochs 150, Optimizer SGD, lr0 0.01; Extra args: cos_lr=True", dict(epochs=150, **SGD_COS)),
    ],
}
STAGE2_OVERRIDES = {}   # fill from Claude's reply after Stage 1, e.g. {"hsv_v": 0.6}

# Optional: folder with corrected val labels (the host's guidelines_v4 from Discord) in a PRIVATE dataset.
VAL_LABELS_OVERRIDE = None   # e.g. "/kaggle/input/my-private-val-v4/labels/val"

In [ ]:
# ---- Environment ------------------------------------------------------------
!pip -q install "ultralytics==8.4.66" pycocotools
import os, sys, json, hashlib, shutil, subprocess, urllib.request, zipfile
from pathlib import Path
import torch, ultralytics
from ultralytics import settings as ul_settings
ul_settings.update({"sync": False})   # no ultralytics analytics from this private run
print("torch", torch.__version__, "| ultralytics", ultralytics.__version__)
assert ultralytics.__version__ == "8.4.66", "the plugin uses 8.4.66; results from another version do not transfer"
N_GPU = torch.cuda.device_count()
assert N_GPU > 0, "No GPU. Settings > Accelerator > GPU T4 x2"
print([torch.cuda.get_device_name(i) for i in range(N_GPU)], "| CPUs:", os.cpu_count())
try:
    urllib.request.urlopen("https://competitions.3lc.ai", timeout=15)
except Exception as e:
    raise SystemExit(f"Internet is off or blocked ({e}). Settings > Internet > On")

In [ ]:
# ---- Starter kit: download to /tmp (not saved in the notebook output), sha256-verified ----
CDN = "https://competitions.3lc.ai/kaggle/exdark-low-light/starter-kit/v3"
ROOT = Path("/tmp/exdark"); DL = ROOT / "kit_dl"; DL.mkdir(parents=True, exist_ok=True)
KIT = ROOT / "starter_kit"

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(1 << 20), b""): h.update(c)
    return h.hexdigest()

counts = lambda: tuple(len(list((KIT / "images" / s).glob("*.jpg"))) for s in ("train", "val", "test"))
if KIT.exists() and counts() == (5910, 733, 715):
    print("kit already present")
else:
    manifest = json.load(urllib.request.urlopen(CDN + "/manifest.json"))
    print("kit", manifest["kit_version"], manifest["file_count"], "files")
    for a in manifest["archives"]:
        part = DL / a["name"]
        for attempt in range(3):
            urllib.request.urlretrieve(f"{CDN}/{a['name']}", part)
            if part.stat().st_size == a["bytes"] and sha256(part) == a["sha256"]:
                break
            print("retry", a["name"])
        else:
            raise SystemExit(f"hash check failed for {a['name']}")
        with zipfile.ZipFile(part) as z: z.extractall(ROOT)
        part.unlink()
    print("extracted")
assert counts() == (5910, 733, 715), f"unexpected image counts {counts()}"
print("train/val/test:", counts())

In [ ]:
# ---- Official checkpoint (same sha256 the plugin pins) and dataset yaml ----
CKPT_URL = "https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11n.pt"
CKPT_SHA = "0ebbc80d4a7680d14987a577cd21342b65ecfd94632bd9a8da63ae6417644ee1"
CKPT = ROOT / "yolo11n.pt"
if not CKPT.exists() or sha256(CKPT) != CKPT_SHA:
    urllib.request.urlretrieve(CKPT_URL, CKPT)
assert sha256(CKPT) == CKPT_SHA, "checkpoint hash mismatch"

NAMES = ["Bicycle", "Boat", "Bottle", "Bus", "Car", "Cat", "Chair", "Cup", "Dog", "Motorbike", "People", "Table"]
DATA_YAML = ROOT / "data.yaml"   # absolute paths: ultralytics mis-resolves the kit's relative 'path: .'
DATA_YAML.write_text(json.dumps({"path": str(KIT), "train": str(KIT / "images/train"),
                                 "val": str(KIT / "images/val"), "nc": 12, "names": NAMES}))
VAL_LABELS = Path(VAL_LABELS_OVERRIDE) if VAL_LABELS_OVERRIDE else KIT / "labels/val"
assert len(list(VAL_LABELS.glob("*.txt"))) == 733, f"expected 733 val label files in {VAL_LABELS}"

# Val images whose scene also appears in train (dHash + ORB audit). Scored separately as 'noleak'.
LEAK_STEMS = ["2015_00448", "2015_00456", "2015_00474", "2015_00645", "2015_02686", "2015_02918", "2015_04404",
              "2015_04934", "2015_05124", "2015_05678", "2015_05720", "2015_06049", "2015_06557", "2015_07176"]
print("ready:", DATA_YAML, "| val labels:", VAL_LABELS)

In [ ]:
%%writefile /kaggle/working/sweep_worker.py
"""One sweep run: train like the 3LC plugin, then score val like Kaggle.

Usage: python sweep_worker.py <config.json>
Config keys: name, out_dir, data_yaml, model, val_images, val_labels,
leak_stems, confs, deadline_ts (optional unix time), plugin (text),
train_args (ultralytics kwargs; only what the plugin accepts).
Writes <out_dir>/<name>/result.json. Skips if result.json exists.
"""
import json, sys, time
from pathlib import Path

LOCKED = {"imgsz": 640, "pretrained": True}
FORBIDDEN = {"model", "imgsz", "pretrained", "weights", "resume",
             "data", "tables", "settings", "project", "name", "exist_ok", "task"}
# Plugin field bounds (trainer._EXPOSED); the plugin checks extra args against them too.
BOUNDS = {"epochs": (1, 300), "batch": (1, 128), "lr0": (0.0001, 0.1),
          "lrf": (0.01, 1.0), "patience": (0, 100)}
PLUGIN_OPTIMIZERS = {"auto", "SGD", "Adam", "AdamW", "NAdam", "RAdam", "RMSProp"}
NC = 12


def check_plugin_legal(args):
    bad = FORBIDDEN & set(args)
    if bad:
        raise ValueError(f"not settable in the plugin: {sorted(bad)}")
    for k, (lo, hi) in BOUNDS.items():
        if k in args and not lo <= args[k] <= hi:
            raise ValueError(f"{k}={args[k]} outside plugin bounds {lo}..{hi}")
    if args.get("optimizer", "auto") not in PLUGIN_OPTIMIZERS:
        raise ValueError(f"optimizer {args['optimizer']} is not in the plugin dropdown")


def load_val(val_images, val_labels):
    """[(stem, path, w, h, [(cls, x, y, w, h) in pixels])] for every val image."""
    from PIL import Image
    out = []
    for p in sorted(Path(val_images).glob("*.jpg")):
        w, h = Image.open(p).size
        gts = []
        lp = Path(val_labels) / f"{p.stem}.txt"
        if lp.exists():
            for line in lp.read_text().split("\n"):
                t = line.split()
                if len(t) >= 5:
                    c, xc, yc, bw, bh = int(float(t[0])), *map(float, t[1:5])
                    gts.append((c, (xc - bw / 2) * w, (yc - bh / 2) * h, bw * w, bh * h))
        out.append((p.stem, p, w, h, gts))
    return out


def predict_all(model, val, conf_min):
    """Plugin-identical inference (imgsz 640, max_det 300, default NMS IoU), once.

    Raising conf afterwards equals predicting at that conf: greedy NMS only lets a
    box be suppressed by a higher-scoring one, and the top-300 cap keeps by score.
    """
    preds = {}
    for i in range(0, len(val), 32):  # plugin predicts in chunks of 16-32 images
        chunk = val[i:i + 32]
        for (stem, _, w, h, _), r in zip(chunk, model.predict([str(v[1]) for v in chunk], imgsz=640,
                                                               conf=conf_min, max_det=300, verbose=False)):
            dets = []
            b = r.boxes
            if b is not None and len(b):
                for c, s, (xc, yc, bw, bh) in zip(b.cls.tolist(), b.conf.tolist(), b.xywhn.tolist()):
                    xc, yc, bw, bh = (min(1.0, max(0.0, v)) for v in (xc, yc, bw, bh))  # plugin clamps
                    dets.append((int(c), float(s), [(xc - bw / 2) * w, (yc - bh / 2) * h, bw * w, bh * h]))
            preds[stem] = dets
    return preds


def kaggle_ap50(val, preds, conf, exclude=frozenset()):
    """pycocotools bbox AP@0.5, maxDets=100, mean over classes that have GT."""
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    import contextlib, io

    images, anns, dets = [], [], []
    for idx, (stem, _, w, h, gts) in enumerate((v for v in val if v[0] not in exclude), 1):
        images.append({"id": idx, "width": w, "height": h})
        for c, x, y, bw, bh in gts:
            anns.append({"id": len(anns) + 1, "image_id": idx, "category_id": c + 1, "iscrowd": 0,
                         "bbox": [x, y, bw, bh], "area": bw * bh})
        dets += [{"image_id": idx, "category_id": c + 1, "score": s, "bbox": bb}
                 for c, s, bb in preds[stem] if s >= conf]
    if not dets:
        return 0.0, {}
    gt = COCO()
    gt.dataset = {"images": images, "annotations": anns,
                  "categories": [{"id": k + 1} for k in range(NC)]}
    with contextlib.redirect_stdout(io.StringIO()):
        gt.createIndex()
        ev = COCOeval(gt, gt.loadRes(dets), "bbox")
        ev.params.iouThrs = [0.5]
        ev.params.maxDets = [100]
        ev.params.areaRng = [ev.params.areaRng[0]]
        ev.params.areaRngLbl = ["all"]
        ev.evaluate(); ev.accumulate()
    prec = ev.eval["precision"][0, :, :, 0, 0]  # [recall, class]; -1 = class has no GT
    per_class = {k: float(prec[:, k][prec[:, k] > -1].mean()) for k in range(NC) if (prec[:, k] > -1).any()}
    return sum(per_class.values()) / len(per_class), per_class


def main():
    cfg = json.loads(Path(sys.argv[1]).read_text())
    out = Path(cfg["out_dir"]) / cfg["name"]
    res_path = out / "result.json"
    if res_path.exists():
        print(f"[{cfg['name']}] already done, skipping")
        return
    args = dict(cfg["train_args"])
    check_plugin_legal(args)

    from ultralytics import YOLO
    import pandas as pd

    deadline = cfg.get("deadline_ts")
    hit_deadline = {"v": False}

    def stop_at_deadline(trainer):
        if deadline and time.time() > deadline:
            hit_deadline["v"] = True
            trainer.stop = True  # ends after this epoch; best.pt/last.pt are still written

    t0 = time.time()
    model = YOLO(cfg["model"])
    model.add_callback("on_fit_epoch_end", stop_at_deadline)
    model.train(data=cfg["data_yaml"], project=str(Path(cfg["out_dir"])), name=cfg["name"],
                exist_ok=True, plots=False, **args, **LOCKED)
    train_s = time.time() - t0

    hist = pd.read_csv(out / "results.csv")
    hist.columns = [c.strip() for c in hist.columns]
    m50, m5095 = hist["metrics/mAP50(B)"], hist["metrics/mAP50-95(B)"]
    ep = lambda i: int(hist["epoch"][i]) + (1 if hist["epoch"].min() == 0 else 0)
    result = {
        "name": cfg["name"], "plugin": cfg.get("plugin", ""), "train_args": args,
        "epochs_run": int(len(hist)), "stopped_by_deadline": hit_deadline["v"],
        "train_minutes": round(train_s / 60, 1),
        "ul_best_map50": round(float(m50.max()), 4), "ul_best_map50_epoch": ep(m50.idxmax()),
        # ultralytics 8.4 saves best.pt by mAP50-95 only; Kaggle scores mAP50
        "ul_bestpt_epoch": ep(m5095.idxmax()), "ul_map50_at_bestpt": round(float(m50[m5095.idxmax()]), 4),
        "ul_last_map50": round(float(m50.iloc[-1]), 4),
        "map50_curve": [round(float(v), 4) for v in m50],
        "kaggle": {},
    }
    val = load_val(cfg["val_images"], cfg["val_labels"])
    leak = frozenset(cfg.get("leak_stems", []))
    for tag in ("best", "last"):
        w = out / "weights" / f"{tag}.pt"
        if not w.exists():
            continue
        preds = predict_all(YOLO(str(w)), val, min(cfg["confs"]))
        for conf in cfg["confs"]:
            full, per_class = kaggle_ap50(val, preds, conf)
            noleak, _ = kaggle_ap50(val, preds, conf, leak)
            result["kaggle"][f"{tag}@{conf}"] = {"ap50": round(full, 4), "ap50_noleak": round(noleak, 4),
                                                 "per_class": {k: round(v, 3) for k, v in per_class.items()}}
    res_path.write_text(json.dumps(result, indent=1))
    print(f"[{cfg['name']}] done in {train_s / 60:.1f} min")


if __name__ == "__main__":
    main()

In [ ]:
# ---- Run the configs, one per GPU at a time, with a hard time budget ----
OUT = Path("/kaggle/working/sweep"); LOGS = OUT / "logs"; LOGS.mkdir(parents=True, exist_ok=True)
DEADLINE = NB_START + TIME_BUDGET_H * 3600
EVAL_MARGIN = 20 * 60   # training stops this long before the deadline so scoring can finish
WORKERS = max(1, (os.cpu_count() or 4) // N_GPU)

configs = []
for name, plugin, args in STAGES[STAGE]:
    a = {**COMMON, **args, **(STAGE2_OVERRIDES if STAGE == 2 else {})}
    configs.append({"name": name, "plugin": plugin + (f" + {STAGE2_OVERRIDES}" if STAGE == 2 and STAGE2_OVERRIDES else ""),
                    "out_dir": str(OUT), "data_yaml": str(DATA_YAML), "model": str(CKPT),
                    "val_images": str(KIT / "images/val"), "val_labels": str(VAL_LABELS),
                    "leak_stems": LEAK_STEMS, "confs": CONFS, "deadline_ts": DEADLINE - EVAL_MARGIN,
                    "train_args": {**a, "device": 0, "workers": WORKERS}})

queue = [c for c in configs if not (OUT / c["name"] / "result.json").exists()]
running, done_minutes, skipped = {}, [], []
print(f"{len(queue)} configs to run on {N_GPU} GPU(s), {WORKERS} dataloader workers each")

def min_per_epoch():
    return max(done_minutes) if done_minutes else None

while queue or running:
    for gpu in range(N_GPU):
        if gpu in running or not queue:
            continue
        c = queue.pop(0)
        est = min_per_epoch()
        if est and time.time() + est * c["train_args"]["epochs"] * 60 > DEADLINE - EVAL_MARGIN:
            print(f"note: {c['name']} may not finish all epochs; it stops at the deadline and is still scored")
        if time.time() > DEADLINE - EVAL_MARGIN - 15 * 60:
            skipped.append(c["name"]); print("skip (no time left):", c["name"]); continue
        cf = OUT / f"{c['name']}.json"; cf.write_text(json.dumps(c))
        log = open(LOGS / f"{c['name']}.log", "w")
        p = subprocess.Popen([sys.executable, "/kaggle/working/sweep_worker.py", str(cf)], stdout=log,
                             stderr=subprocess.STDOUT, env={**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)},
                             cwd=str(OUT))
        running[gpu] = (p, c, time.time(), log)
        print(time.strftime("%H:%M"), f"GPU{gpu} start", c["name"])
        time.sleep(60)   # stagger so two processes do not download ultralytics' AMP-check file at once
    time.sleep(30)
    for gpu, (p, c, t0, log) in list(running.items()):
        if p.poll() is None:
            continue
        log.close(); del running[gpu]
        res = OUT / c["name"] / "result.json"
        if p.returncode == 0 and res.exists():
            r = json.loads(res.read_text())
            done_minutes.append(r["train_minutes"] / max(1, r["epochs_run"]))
            print(time.strftime("%H:%M"), f"GPU{gpu} done ", c["name"], f"{r['train_minutes']} min,",
                  f"{r['epochs_run']} epochs, ul mAP50 {r['ul_best_map50']}")
        else:
            print(time.strftime("%H:%M"), f"GPU{gpu} FAILED", c["name"], "exit", p.returncode)
            print("".join(open(LOGS / f"{c['name']}.log").readlines()[-25:]))
print("all finished; skipped:", skipped)

In [ ]:
# ---- Summary ------------------------------------------------------------------
import pandas as pd
results = [json.loads(p.read_text()) for p in sorted(OUT.glob("*/result.json"))]
assert results, "no finished runs"

def best_conf(r, tag):
    keys = [k for k in r["kaggle"] if k.startswith(tag + "@")]
    if not keys:
        return None, {"ap50": None, "ap50_noleak": None, "per_class": {}}
    k = max(keys, key=lambda k: r["kaggle"][k]["ap50"])
    return float(k.split("@")[1]), r["kaggle"][k]

rows = []
for r in results:
    bc, b = best_conf(r, "best"); lc, l = best_conf(r, "last")
    rows.append({"name": r["name"], "epochs": r["epochs_run"], "min": r["train_minutes"],
                 "kaggle_best.pt": b["ap50"], "at_conf": bc, "noleak": b["ap50_noleak"],
                 "best.pt@0.25": r["kaggle"].get("best@0.25", {}).get("ap50"),
                 "kaggle_last.pt": l["ap50"], "ul_best_map50": r["ul_best_map50"],
                 "ul_peak_ep": r["ul_best_map50_epoch"], "bestpt_ep": r["ul_bestpt_epoch"],
                 "deadline_stop": r["stopped_by_deadline"]})
df = pd.DataFrame(rows).sort_values("kaggle_best.pt", ascending=False)
pd.set_option("display.width", 250); print(df.to_string(index=False))

by = {r["name"]: r for r in results}
if "s1_sgd_cos" in by and "s1_sgd_cos_seed1" in by:
    noise = abs(best_conf(by["s1_sgd_cos"], "best")[1]["ap50"] - best_conf(by["s1_sgd_cos_seed1"], "best")[1]["ap50"])
    print(f"\nseed noise (same config, seed 0 vs 1): {noise:.4f}. Treat smaller gaps as no difference.")

top = by[df.iloc[0]["name"]]
print(f"\nbest config: {top['name']}\n  plugin: {top['plugin']}")
tc, tv = best_conf(top, "best")
print("  per-class AP50 at conf", tc, {NAMES[int(k)]: v for k, v in tv["per_class"].items()})

compact = []
for r in results:
    compact.append({"name": r["name"], "epochs_run": r["epochs_run"], "min": r["train_minutes"],
                    "deadline": r["stopped_by_deadline"], "ul_best": [r["ul_best_map50"], r["ul_best_map50_epoch"]],
                    "bestpt_ep": r["ul_bestpt_epoch"], "ul_last": r["ul_last_map50"],
                    "curve_every5": r["map50_curve"][4::5],
                    "kaggle": {k: [v["ap50"], v["ap50_noleak"]] for k, v in r["kaggle"].items()}})
print("\n=== PASTE EVERYTHING BELOW ===")
print(json.dumps({"stage": STAGE, "gpus": [torch.cuda.get_device_name(i) for i in range(N_GPU)],
                  "per_class_top": {"name": top["name"], "conf": tc,
                                    "ap50": {NAMES[int(k)]: v for k, v in tv["per_class"].items()}},
                  "runs": compact}, separators=(",", ":")))